<a href="https://colab.research.google.com/github/DerikVo/COOP_Teaching_Material/blob/main/Big_query_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
subprocess.run(["pip", "install", "--quiet", "gdown", "google-cloud-bigquery", "pandas", "pyarrow"])

from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import gdown
import random
import os

auth.authenticate_user()


In [ ]:
def csv_to_table_name(filename):
    base = filename.replace(".csv", "").replace("-divvy-tripdata", "")
    return f"trips_{base}"

In [ ]:
DATASET_ID      = "cyclistic_data_2025"
LOCATION        = "US"
DRIVE_FOLDER_ID = "1UwIm5RA99Nk1A2cErmNaEMLnBrDI30_A"
CSV_FILES = [
    "202501-divvy-tripdata.csv",
    "202502-divvy-tripdata.csv",
    "202503-divvy-tripdata.csv",
]

In [ ]:
MONTH_MAP = {
    "202501-divvy-tripdata.csv": "jan_2025",
    "202502-divvy-tripdata.csv": "feb_2025",
    "202503-divvy-tripdata.csv": "mar_2025",
}

In [ ]:
PROJECT_ID = f"divvy-trips-{random.randint(10000, 99999)}"

subprocess.run(["gcloud", "projects", "create", PROJECT_ID], check=True)
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)
subprocess.run(["gcloud", "services", "enable", "bigquery.googleapis.com", "--project", PROJECT_ID], check=True)

print(f"Project created: {PROJECT_ID}")

Project created: divvy-trips-81391


In [ ]:
client = bigquery.Client(project=PROJECT_ID)
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET_ID}")

Dataset ready: divvy-trips-81391.cyclistic_data_2025


In [ ]:
output_dir = "/content/divvy_data"
os.makedirs(output_dir, exist_ok=True)
gdown.download_folder(
    f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}",
    output=output_dir,
    quiet=False,
    use_cookies=False
)

Retrieving folder contents


Processing file 1xOKtUYSPKDpEgsoxCe5jx51vWyeH96cI 202501-divvy-tripdata.csv
Processing file 1JqUlZK992snI7J09GPENUp0p1pwf5mDc 202502-divvy-tripdata.csv
Processing file 1m6IQx40we5G_DQc-h7SUrxbijGWAzxP2 202503-divvy-tripdata.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1xOKtUYSPKDpEgsoxCe5jx51vWyeH96cI
To: /content/divvy_data/202501-divvy-tripdata.csv
100%|██████████| 28.6M/28.6M [00:00<00:00, 109MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JqUlZK992snI7J09GPENUp0p1pwf5mDc
To: /content/divvy_data/202502-divvy-tripdata.csv
100%|██████████| 31.3M/31.3M [00:00<00:00, 81.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1m6IQx40we5G_DQc-h7SUrxbijGWAzxP2
To: /content/divvy_data/202503-divvy-tripdata.csv
100%|██████████| 60.9M/60.9M [00:00<00:00, 123MB/s]
Download completed


['/content/divvy_data/202501-divvy-tripdata.csv',
 '/content/divvy_data/202502-divvy-tripdata.csv',
 '/content/divvy_data/202503-divvy-tripdata.csv']

In [ ]:
job_config = bigquery.LoadJobConfig(
    autodetect=True,
    skip_leading_rows=1,
    source_format=bigquery.SourceFormat.CSV,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

for filename in CSV_FILES:
    filepath = os.path.join(output_dir, filename)
    if not os.path.exists(filepath):
        print(f"File not found, skipping: {filename}")
        continue

    table_id = f"{PROJECT_ID}.{DATASET_ID}.{MONTH_MAP[filename]}"
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    load_job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    load_job.result()

    table = client.get_table(table_id)
    print(f"{table_id}: {table.num_rows:,} rows loaded")

divvy-trips-81391.cyclistic_data_2025.jan_2025: 138,688 rows loaded
divvy-trips-81391.cyclistic_data_2025.feb_2025: 151,879 rows loaded
divvy-trips-81391.cyclistic_data_2025.mar_2025: 298,154 rows loaded


In [ ]:
for t in client.list_tables(f"{PROJECT_ID}.{DATASET_ID}"):
    full_table = client.get_table(t.reference)
    print(f"{t.table_id}: {full_table.num_rows:,} rows")

feb_2025: 151,879 rows
jan_2025: 138,688 rows
mar_2025: 298,154 rows
